In [13]:
# Importing required dependencies
from paddleocr import PaddleOCR
from IPython.display import display, Markdown, HTML
import cv2
import numpy as np
import os

In [14]:
def initialize_ocr():
    params = {"lang": "en"}
    strategies = [
        {"use_angle_cls": True},
        {"use_textline_orientation": True},
        {}
    ]
    
    for strategy in strategies:
        try:
            return PaddleOCR(**{**params, **strategy})
        except:
            continue
    return None

ocr = initialize_ocr()
if ocr:
    display(Markdown("### OCR Engine Ready"))

C:\Users\sugam\AppData\Local\Temp\ipykernel_21836\989118998.py:11: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  return PaddleOCR(**{**params, **strategy})
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sugam\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sugam\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sugam\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please dele

### OCR Engine Ready

In [ ]:

# Image for testing
img_path = 'image.png'

if os.path.exists(img_path):
    try:
        display(Markdown(f"🔍 Scanning **{img_path}** with Edge-Padding..."))
        
        # --- PRE-PROCESSING STEP ---
        # Load original image
        original_img = cv2.imread(img_path)
        
        # Add white padding (50px) to all sides to fix edge detection issues
        # This gives the engine more 'spatial context' for text at boundaries
        padded_img = cv2.copyMakeBorder(
            original_img, 
            50, 50, 50, 50, 
            cv2.BORDER_CONSTANT, 
            value=[255, 255, 255]
        )
        # ---------------------------

        # Result capture
        try:
            result = ocr.predict(padded_img)
        except Exception as e:
            result = ocr.ocr(padded_img, cls=True)
            
        if result:
            # Extract data
            final_data = []
            if isinstance(result[0], dict):
                res_dict = result[0]
                final_data = list(zip(res_dict.get('rec_texts', []), res_dict.get('rec_scores', [])))
            else:
                current = result[0] if isinstance(result[0], list) else result
                final_data = [line[1] for line in current if isinstance(line, list) and len(line) > 1]
            
            # UI Rendering
            output_html = f"<div style='background-color:#f8f9fa; padding:20px; border-radius:15px; border:1px solid #dee2e6;'>"
            output_html += f"<h2 style='color:#2c3e50; border-bottom:2px solid #3498db; padding-bottom:10px;'>📄 Document Content</h2>"
            
            for text, score in final_data:
                color = "#27ae60" if score > 0.9 else "#f39c12" if score > 0.7 else "#c0392b"
                output_html += f"""
                <div style='margin-bottom:12px; display:flex; align-items:center;'>
                    <span style='background-color:{color}; color:white; padding:2px 8px; border-radius:10px; font-size:12px; font-weight:bold; margin-right:15px; min-width:50px; text-align:center;'>
                        {score*100:.1f}%
                    </span>
                    <span style='font-family:Segoe UI, sans-serif; font-size:16px; color:#2c3e50;'>{text}</span>
                </div>
                """
            
            output_html += "</div>"
            display(HTML(output_html))
            
            if final_data:
                avg_acc = sum([d[1] for d in final_data]) / len(final_data)
                display(Markdown(f"---\n**Summary:** {len(final_data)} lines detected | **Average Confidence:** {avg_acc*100:.1f}%"))
            
        else:
            display(Markdown("No text detected. Image might be too blurry or incorrect format."))
            
    except Exception as e:
        print(f"Extraction error: {e}")
else:
    display(Markdown(f"ERROR: File **{img_path}** not found."))

🔍 Scanning **image.png** with Edge-Padding...

---
**Summary:** 5 lines detected | **Average Confidence:** 99.6%